# 03 - Agente LangGraph versionado e rastreado com MLflow

Caso prático: um **atendente** de loja com um grafo LangGraph de três nós. Cada nó que chama o LLM usa um prompt do Prompt Registry.

Fluxo:

```
START → classificar ──(devolucao ou entrega)──→ responder → END
            │
            └────────────(outro)──────────────→ fora_do_escopo → END
```

- **`classificar`**: usa o prompt `langgraph-classificador` e responde com uma categoria.
- **`responder`**: usa o prompt `langgraph-responder`, preenchido com a categoria e a pergunta.
- **`fora_do_escopo`**: resposta fixa, sem LLM.

O notebook faz, nesta ordem:

1. registra os dois prompts no MLflow;
2. monta o grafo a partir do código em `agente/agente_langgraph.py`, que usa os prompts registrados;
3. testa o grafo localmente;
4. registra o agente como modelo do MLflow (`agente-langgraph`), com os prompts ligados ao modelo;
5. invoca o agente pelo modelo registrado e mostra os traces gravados no experimento;
6. no fim, mostra como chamar o agente servido com `curl`.

Cada execução do notebook cria versões novas de prompts e de modelo. Veja a seção de observações no fim.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto. O experimento `agente-langgraph` é separado dos anteriores. O banco é o mesmo `mlflow.db` da pasta.

In [1]:
import os
from pathlib import Path

import mlflow
import yaml
from dotenv import find_dotenv, load_dotenv
from IPython.display import Code, Markdown, display
from langchain_core.messages import HumanMessage
from mlflow import MlflowClient

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")

EXPERIMENTO = "agente-langgraph"
NOME_MODELO = "agente-langgraph"
NOME_PROMPT_CLASSIFICADOR = "langgraph-classificador"
NOME_PROMPT_RESPONDER = "langgraph-responder"

ARQUIVO_CODIGO = Path("agente/agente_langgraph.py")
PASTA_PROMPTS = Path("prompts")
PASTA_PROMPTS.mkdir(exist_ok=True)

experimento = mlflow.set_experiment(EXPERIMENTO)
print(f"Experimento '{experimento.name}' (id {experimento.experiment_id})")

Experimento 'agente-langgraph' (id 1)


## 2. Registrando os prompts

Cada prompt é um YAML com `commit`, `system` e `user`. `registrar_prompt` converte o YAML em mensagens de chat e registra no Prompt Registry, com o nome indicado. O prompt do classificador recebe `{{ pergunta }}`. O prompt do responder recebe `{{ nome_loja }}`, `{{ categoria }}` e `{{ pergunta }}`.

In [2]:
YAML_CLASSIFICADOR = """
commit: "Classifica a pergunta em devolucao, entrega ou outro"
system: |
  Você classifica perguntas de clientes de uma loja. Responda apenas com uma palavra: devolucao, entrega ou outro.
  Use devolucao para trocas, devoluções e reembolsos. Use entrega para prazos e rastreio de pedidos.
  Use outro para qualquer outro assunto.
user: |
  {{ pergunta }}
"""

YAML_RESPONDER = """
commit: "Responde à pergunta de acordo com a categoria"
system: |
  Você é o atendente virtual da {{ nome_loja }}. Responda em português, de forma cordial,
  em até 3 frases. Assunto da pergunta: {{ categoria }}.
user: |
  {{ pergunta }}
"""

for nome_arquivo, conteudo in [
    ("langgraph_classificador.yaml", YAML_CLASSIFICADOR),
    ("langgraph_responder.yaml", YAML_RESPONDER),
]:
    (PASTA_PROMPTS / nome_arquivo).write_text(conteudo, encoding="utf-8")


def registrar_prompt(caminho: Path, nome: str):
    dados = yaml.safe_load(caminho.read_text(encoding="utf-8"))
    mensagens = [
        {"role": "system", "content": dados["system"]},
        {"role": "user", "content": dados["user"]},
    ]
    return mlflow.genai.register_prompt(name=nome, template=mensagens, commit_message=dados["commit"])


prompt_classificador = registrar_prompt(PASTA_PROMPTS / "langgraph_classificador.yaml", NOME_PROMPT_CLASSIFICADOR)
prompt_responder = registrar_prompt(PASTA_PROMPTS / "langgraph_responder.yaml", NOME_PROMPT_RESPONDER)

URI_CLASSIFICADOR = f"prompts:/{NOME_PROMPT_CLASSIFICADOR}/{prompt_classificador.version}"
URI_RESPONDER = f"prompts:/{NOME_PROMPT_RESPONDER}/{prompt_responder.version}"
print(URI_CLASSIFICADOR)
print(URI_RESPONDER)

prompts:/langgraph-classificador/2
prompts:/langgraph-responder/2


## 3. Código do agente

O grafo é construído em `agente/agente_langgraph.py`. O arquivo carrega os prompts pelas URIs fixas, define os três nós e as arestas, e chama `set_model`. O `mlflow.langchain.autolog()` no topo faz o trace de cada invocação. Veja o código:

In [3]:
display(Code(filename=str(ARQUIVO_CODIGO), language="python"))

"""Agente LangGraph versionado com MLflow (models from code).

O grafo tem três nós:
- classificar: escolhe a categoria da pergunta (usa o prompt `langgraph-classificador`);
- responder: responde à pergunta de devolução ou entrega (usa o prompt `langgraph-responder`);
- fora_do_escopo: responde com uma mensagem fixa quando a pergunta não é atendida.

As URIs dos prompts ficam fixas aqui, então cada versão do agente usa versões conhecidas.
O `autolog` do LangChain grava um trace a cada invocação.
"""

import os

import mlflow
from langchain_core.messages import AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessagesState, StateGraph
from mlflow.models import set_model

PROMPT_CLASSIFICADOR = "prompts:/langgraph-classificador/1"
PROMPT_RESPONDER = "prompts:/langgraph-responder/1"
NOME_LOJA = "Loja Exemplo"
CATEGORIAS_ATENDIDAS = ("devolucao", "entrega")

mlflow.langchain.autolog()

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), api_key=os.getenv("OPENAI_API_KEY"))

prompt_classificador = mlflow.genai.load_prompt(PROMPT_CLASSIFICADOR)
prompt_responder = mlflow.genai.load_prompt(PROMPT_RESPONDER)


class Estado(MessagesState):
    categoria: str


def pergunta_atual(estado: Estado) -> str:
    return estado["messages"][-1].content


def classificar(estado: Estado) -> dict:
    mensagens = prompt_classificador.format(pergunta=pergunta_atual(estado))
    categoria = llm.invoke(mensagens).content.strip().lower()
    if categoria not in CATEGORIAS_ATENDIDAS:
        categoria = "outro"
    return {"categoria": categoria}


def responder(estado: Estado) -> dict:
    mensagens = prompt_responder.format(
        nome_loja=NOME_LOJA,
        categoria=estado["categoria"],
        pergunta=pergunta_atual(estado),
    )
    resposta = llm.invoke(mensagens)
    return {"messages": [AIMessage(content=resposta.content)]}


def fora_do_escopo(estado: Estado) -> dict:
    texto = "Só consigo ajudar com devoluções e entregas. Vou encaminhar sua pergunta a um atendente."
    return {"messages": [AIMessage(content=texto)]}


def rotear(estado: Estado) -> str:
    return "responder" if estado["categoria"] in CATEGORIAS_ATENDIDAS else "fora_do_escopo"


construtor = StateGraph(Estado)
construtor.add_node("classificar", classificar)
construtor.add_node("responder", responder)
construtor.add_node("fora_do_escopo", fora_do_escopo)

construtor.add_edge(START, "classificar")
construtor.add_conditional_edges("classificar", rotear, ["responder", "fora_do_escopo"])
construtor.add_edge("responder", END)
construtor.add_edge("fora_do_escopo", END)

grafo_agente = construtor.compile()

set_model(grafo_agente)

## 4. Testando o grafo localmente

Antes de registrar, rodamos o grafo direto. A entrada é uma lista de mensagens. A saída é o estado final, com a categoria escolhida e as mensagens.

In [4]:
from agente.agente_langgraph import grafo_agente

PERGUNTAS = [
    "Posso devolver um produto depois de 10 dias?",
    "Quanto tempo leva para chegar uma entrega em Curitiba?",
    "Qual é a capital da França?",
]

for pergunta in PERGUNTAS:
    estado = grafo_agente.invoke({"messages": [HumanMessage(content=pergunta)]})
    print(f"Pergunta: {pergunta}")
    print(f"Categoria: {estado['categoria']}")
    print(f"Resposta: {estado['messages'][-1].content}\n")

Pergunta: Posso devolver um produto depois de 10 dias?
Categoria: devolucao
Resposta: Se a compra foi a distância (internet/telefone), o Código de Defesa do Consumidor garante o direito de arrependimento por 7 dias a partir do recebimento; após 10 dias esse direito não é obrigatório. Na Loja Exemplo, devoluções fora desse prazo dependem da nossa política interna ou só são aceitas em caso de defeito dentro da garantia; envie seu número do pedido que eu verifico para você. Deseja que eu consulte seu pedido?

Pergunta: Quanto tempo leva para chegar uma entrega em Curitiba?
Categoria: entrega
Resposta: Normalmente entregas para Curitiba levam de 2 a 5 dias úteis com frete padrão; no frete expresso o prazo costuma ser de 1 a 2 dias úteis. O prazo pode variar conforme disponibilidade do produto e seu CEP. Quer me informar o CEP para eu checar o prazo exato?

Pergunta: Qual é a capital da França?
Categoria: outro
Resposta: Só consigo ajudar com devoluções e entregas. Vou encaminhar sua pergun

## 5. Registrando o agente no MLflow

`mlflow.langchain.log_model` recebe o arquivo do grafo (*models from code*), registra uma versão de `agente-langgraph` e liga os prompts ao modelo pela lista `prompts`. Assim o MLflow sabe quais versões de prompt o agente usa.

O exemplo de entrada (`input_example`) define o formato do modelo: uma lista de mensagens sob a chave `messages`.

In [5]:
EXEMPLO = {"messages": [{"role": "user", "content": "Posso devolver um produto depois de 10 dias?"}]}

with mlflow.start_run(run_name="agente-langgraph-v1"):
    mlflow.log_params({
        "prompt_classificador": URI_CLASSIFICADOR,
        "prompt_responder": URI_RESPONDER,
        "codigo": str(ARQUIVO_CODIGO),
    })
    mlflow.set_tag("descricao", "Grafo de três nós com dois prompts do registro")

    info = mlflow.langchain.log_model(
        lc_model=str(ARQUIVO_CODIGO.resolve()),
        name="agente",
        registered_model_name=NOME_MODELO,
        prompts=[URI_CLASSIFICADOR, URI_RESPONDER],
        input_example=EXEMPLO,
    )

print(f"Agente registrado: {NOME_MODELO} versão {info.registered_model_version}")

2026/11/16 22:36:04 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/16 22:36:04 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-16efbe472cd44e46850ee31b8dd2d8ec
2026/11/16 22:36:04 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/11/16 22:36:10 INFO mlflow.models.model: Found the following environment variables used during model inference: [OPENAI_API_KEY]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


Agente registrado: agente-langgraph versão 2


Registered model 'agente-langgraph' already exists. Creating a new version of this model...
Created version '2' of model 'agente-langgraph'.


## 6. Invocando o agente registrado

Carregamos a versão registrada com `mlflow.pyfunc.load_model` e chamamos `predict`. A chamada passa pelo mesmo grafo que testamos antes, agora a partir do modelo do registro.

In [6]:
modelo = mlflow.pyfunc.load_model(f"models:/{NOME_MODELO}/{info.registered_model_version}")

def texto_da_resposta(resposta) -> str:
    estado = resposta[0] if isinstance(resposta, list) else resposta
    return estado["messages"][-1]["content"]


for pergunta in PERGUNTAS:
    resposta = modelo.predict({"messages": [{"role": "user", "content": pergunta}]})
    print(f"Pergunta: {pergunta}")
    print(f"Resposta: {texto_da_resposta(resposta)}\n")

Pergunta: Posso devolver um produto depois de 10 dias?
Resposta: Se a compra foi feita pela internet, telefone ou fora do estabelecimento, o Código de Defesa do Consumidor prevê 7 dias corridos, a contar do recebimento, para desistência e devolução sem custo. Depois de 10 dias só aceitamos devolução em caso de defeito coberto pela garantia ou quando a política comercial da Loja Exemplo permitir; consulte nossa política de devolução ou contate o SAC com o número do pedido. Quer que eu verifique seu pedido agora?

Pergunta: Quanto tempo leva para chegar uma entrega em Curitiba?
Resposta: Olá! O prazo de entrega para Curitiba varia conforme o tipo de frete e a disponibilidade: normalmente 1–3 dias úteis para opções expressas e 3–7 dias úteis para frete econômico. Se quiser, me informe o CEP ou o número do pedido que eu verifico o prazo exato.

Pergunta: Qual é a capital da França?
Resposta: Só consigo ajudar com devoluções e entregas. Vou encaminhar sua pergunta a um atendente.



## 7. Rastreamento no MLflow

Cada invocação gera um trace no experimento `agente-langgraph`. Os spans de primeiro nível mostram os nós do grafo. Abaixo, listamos os traces e os spans do mais recente.

In [7]:
traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)
print(f"Traces registrados no experimento: {len(traces)}")

trace = mlflow.get_trace(traces.iloc[0]["trace_id"])
print(f"\nSpans do trace mais recente ({trace.info.trace_id}):")
for span in trace.data.spans:
    print(f"- {span.name} ({span.span_type})")

Traces registrados no experimento: 10

Spans do trace mais recente (tr-bdb6f6830839d9c774a676efa4542628):
- LangGraph (CHAIN)
- classificar (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- rotear (CHAIN)
- fora_do_escopo (CHAIN)


/tmp/ipykernel_159791/313133131.py:1: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)


## 8. Servindo o agente e chamando com curl

O servidor usa o modelo registrado. Ele precisa de três coisas do ambiente: a chave da OpenAI, a URI do banco e o nome do experimento onde os traces são gravados. As seções a seguir mostram os comandos em dois terminais.

### Terminal 1: subir o servidor

```bash
cd notebooks/03-MLFlow
set -a && source ../../.env && set +a
MLFLOW_TRACKING_URI=sqlite:///mlflow.db MLFLOW_EXPERIMENT_NAME=agente-langgraph \
  uv run --project ../.. mlflow models serve -m models:/agente-langgraph/1 -p 5003 --env-manager local
```

- `MLFLOW_TRACKING_URI`: onde estão o modelo e os prompts do registro.
- `MLFLOW_EXPERIMENT_NAME`: experimento onde o servidor grava os traces de cada chamada.
- `-m models:/agente-langgraph/1`: a versão do agente. Troque o número para servir outra versão.
- `-p 5003`: a porta. Ela precisa estar livre.

Quando aparecer `Uvicorn running on http://127.0.0.1:5003`, o servidor está pronto. Para parar, pressione `Ctrl+C`.

### Terminal 2: chamando com curl

Confira se o servidor está no ar:

```bash
curl http://127.0.0.1:5003/health
```

Envie uma pergunta:

```bash
curl -s http://127.0.0.1:5003/invocations \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Posso devolver um produto depois de 10 dias?"}]}'
```

Para ver só o texto da resposta:

```bash
curl -s http://127.0.0.1:5003/invocations \
  -H "Content-Type: application/json" \
  -d '{"messages": [{"role": "user", "content": "Quanto tempo leva para chegar uma entrega em Curitiba?"}]}' \
  | python3 -c "import sys, json; print(json.load(sys.stdin)[0]['messages'][-1]['content'])"
```

Cada chamada gera um trace novo no experimento `agente-langgraph`, que pode ser visto na interface do MLflow.

## 9. Observações

- **Reexecução**: cada execução cria novas versões de prompt e novas versões do modelo `agente-langgraph`. Para recomeçar, apague `mlflow.db` e a pasta `prompts/`.
- **Interface**: com `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000` rodando na pasta do notebook, o experimento `agente-langgraph` mostra os runs e os traces, e a seção Prompts mostra os dois prompts.
- **Prompts ligados ao modelo**: a lista `prompts` do `log_model` registra quais versões o agente usa. Para trocar um prompt, registre uma nova versão e atualize as URIs em `agente/agente_langgraph.py`.